In [1]:
import pandas as pd
import json
import os

# 1. CSV 파일 읽기 (인증코딩 'cp949' 적용)
csv_file = '충청남도_전국관광지정보_20240709.csv'
df = pd.read_csv(csv_file, encoding='cp949')

# 2. 주소에서 '시/군' 이름 추출하는 함수
def extract_sigungu(row):
    addr = str(row.get('소재지도로명주소', ''))
    if '충청남도' not in addr:
        addr = str(row.get('소재지지번주소', ''))
    
    parts = addr.split()
    for part in parts:
        if part.endswith(('시', '군', '구')):
            return part
    return '기타'

df['시군명'] = df.apply(extract_sigungu, axis=1)

# 3. 컴포넌트에서 쓰기 좋은 형태로 데이터 구조 가공
records = []
for idx, row in df.iterrows():
    record = {
        "연번": idx + 1,
        "시군명": row['시군명'],
        "관광지명": row['관광지명'],
        "관광지구분": row.get('관광지구분', '관광지'),
        "관광지 주소": row.get('소재지도로명주소') if pd.notna(row.get('소재지도로명주소')) else row.get('소재지지번주소', ''),
        "관광지 연락처": row.get('관리기관전화번호') if pd.notna(row.get('관리기관전화번호')) else None,
        "관광지소개": row.get('관광지소개') if pd.notna(row.get('관광지소개')) else '',
        "주차가능수": int(row['주차가능수']) if pd.notna(row['주차가능수']) else 0,
        "수용인원수": int(row['수용인원수']) if pd.notna(row['수용인원수']) else 0,
        "편익시설": row.get('공공편익시설정보') if pd.notna(row.get('공공편익시설정보')) else ''
    }
    records.append(record)

# 4. /src/data 디렉토리 생성 및 JSON 저장
output_dir = 'src/data'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'chungnam_tours_detailed.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"성공적으로 변환 완료! 저장 경로: {output_path}")

성공적으로 변환 완료! 저장 경로: src/data\chungnam_tours_detailed.json
